# Administration système avec la bibliothèque standard

Ces exercices pratiquent les modules de la bibliothèque standard vus en cours : `pathlib`, `shutil` et `tempfile` pour les fichiers, `os.environ` pour l'environnement, `sys` et `fileinput` pour les flux standards, `subprocess` pour les commandes externes.

Pour ne rien modifier sur la machine, les exercices travaillent sur une copie miniature d'un serveur, le dossier `serveur/`, que crée la cellule suivante dans le répertoire courant : exécutez-la avant de commencer (l'exécuter à nouveau remet le dossier à zéro). Le notebook tourne dans Colab comme dans le Codespace, qui sont tous deux des Linux Ubuntu où vous êtes `root`. Les cellules qui commencent par `!` lancent des commandes du shell, et l'exercice « Lancer des commandes » des commandes Linux (`dpkg`, `df`, `ls`) : sur votre propre machine, il faut un Linux Debian ou Ubuntu.

Les exercices vont du plus simple au plus difficile : les derniers sont facultatifs si vous manquez de temps.

In [ ]:
import gzip
import os
import shutil
from datetime import UTC, datetime
from pathlib import Path

SERVEUR = Path("serveur")

FICHIERS = {
  "etc/hostname": "web01\n",
  "etc/hosts": "127.0.0.1 localhost\n10.0.0.12 web01\n",
  "etc/logrotate.conf": "weekly\nrotate 4\ncompress\n",
  "etc/nginx/nginx.conf": "user www-data;\ninclude /etc/nginx/conf.d/*.conf;\n",
  "etc/nginx/mime.types": "types {\n  text/html html;\n  text/css css;\n}\n",
  "etc/nginx/conf.d/shop.conf": "server {\n  server_name shop.example.com;\n}\n",
  "etc/nginx/conf.d/blog.conf": "server {\n  server_name blog.example.com;\n}\n",
  "etc/ssh/sshd_config": "Include /etc/ssh/sshd_config.d/*.conf\nPermitRootLogin no\n",
  "etc/ssh/sshd_config.d/hardening.conf": "PasswordAuthentication no\nMaxAuthTries 3\n",
  "etc/systemd/journald.conf": "[Journal]\nStorage=persistent\n",
  "etc/cron.d/backup": "0 3 * * * root /opt/backup/run.sh\n",
  "home/LISEZMOI.txt": "Un dossier par utilisateur.\n",
  "home/alice/.bashrc": "export EDITOR=vim\n",
  "home/alice/notes.txt": "Renouveler le certificat de la boutique.\n",
  "home/bob/rapport.md": "# Rapport d'incident\n",
  "home/carole/.bashrc": "alias ll='ls -l'\n",
  "home/carole/.ssh/authorized_keys": "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5 carole@portable\n",
  "var/log/syslog": """\
Oct 12 06:25:01 web01 CRON[2210]: (root) CMD (/opt/backup/run.sh)
Oct 12 06:25:03 web01 systemd[1]: Started nginx.service - A high performance web server.
Oct 12 07:02:11 web01 nginx[2301]: error: upstream timed out while reading response header
Oct 12 07:02:12 web01 kernel: [ 4012.551] eth0: link up
Oct 12 08:15:40 web01 nginx[2301]: error: connect() failed (111: Connection refused)
Oct 12 09:00:01 web01 systemd[1]: logrotate.service: Deactivated successfully.
Oct 12 10:31:57 web01 sshd[3120]: error: maximum authentication attempts exceeded for root
""",
  "var/log/auth.log": """\
Oct 12 10:30:02 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:30:05 web01 sshd[3118]: Failed password for root from 203.0.113.7 port 52144 ssh2
Oct 12 10:31:57 web01 sshd[3120]: Failed password for invalid user admin from 203.0.113.7
Oct 12 11:12:40 web01 sshd[3302]: Accepted publickey for carole from 192.0.2.15 port 40112
Oct 12 11:12:40 web01 sshd[3302]: pam_unix(sshd:session): session opened for user carole
""",
}

# Les journaux de dpkg, le gestionnaire de paquets, avec leur date de dernière modification
JOURNAUX_DPKG = {
  "dpkg.log": (
    datetime(2026, 10, 12, 6, 25, 3, tzinfo=UTC),
    "2026-10-12 06:25:03 status installed nginx:amd64 1.24.0-2ubuntu7\n",
  ),
  "dpkg.log.1": (
    datetime(2026, 10, 5, 6, 25, 41, tzinfo=UTC),
    "2026-10-05 06:25:41 status installed openssl:amd64 3.0.13-0ubuntu3\n",
  ),
  "dpkg.log.2.gz": (
    datetime(2026, 9, 28, 6, 26, 10, tzinfo=UTC),
    "2026-09-28 06:26:10 status installed curl:amd64 8.5.0-2ubuntu10\n",
  ),
  "dpkg.log.3.gz": (
    datetime(2026, 9, 21, 6, 25, 57, tzinfo=UTC),
    "2026-09-21 06:25:57 status installed git:amd64 1:2.43.0-1ubuntu7\n",
  ),
}

shutil.rmtree(SERVEUR, ignore_errors=True)
for nom, contenu in FICHIERS.items():
  chemin = SERVEUR / nom
  chemin.parent.mkdir(parents=True, exist_ok=True)
  chemin.write_text(contenu, encoding="utf8")
(SERVEUR / "etc" / "nginx" / "sites-enabled").mkdir()
for nom, (date, contenu) in JOURNAUX_DPKG.items():
  chemin = SERVEUR / "var" / "log" / nom
  donnees = contenu.encode()
  if nom.endswith(".gz"):
    donnees = gzip.compress(donnees, mtime=0)
  chemin.write_bytes(donnees)
  os.utime(chemin, (date.timestamp(), date.timestamp()))

for chemin in sorted(SERVEUR.rglob("*")):
  print(chemin)

## Comptes utilisateurs

Le dossier `serveur/home` contient un dossier par utilisateur, et un fichier `LISEZMOI.txt`. Écrivez une fonction `utilisateurs(home)` qui reçoit ce dossier sous forme de `Path` et renvoie la liste triée des noms d'utilisateurs, c'est-à-dire des noms de ses sous-dossiers. Affichez ensuite, pour chaque utilisateur, si son dossier contient un fichier `.bashrc`.

**Documentation utile**

- [`Path.is_dir`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.is_dir)
- [`Path.exists`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.exists)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`iterdir()` donne les éléments du dossier, fichiers compris : `is_dir()` ne garde que les dossiers. `.name` donne le dernier élément d'un chemin, et `sorted` trie. Le chemin d'un fichier de l'utilisateur s'obtient avec l'opérateur `/`, et `exists()` dit s'il existe. Vous devez trouver `['alice', 'bob', 'carole']`, et un `.bashrc` chez alice et carole seulement.

```python
from pathlib import Path


def utilisateurs(home: Path) -> list[str]:
  return sorted(... for element in home.iterdir() if ...)


home = Path("serveur/home")
for nom in utilisateurs(home):
  bashrc = ...
  print(f"{nom} : .bashrc {'présent' if ... else 'absent'}")
```

### Solution

In [ ]:
from pathlib import Path


def utilisateurs(home: Path) -> list[str]:
  return sorted(element.name for element in home.iterdir() if element.is_dir())


home = Path("serveur/home")
print(utilisateurs(home))
for nom in utilisateurs(home):
  bashrc = home / nom / ".bashrc"
  print(f"{nom} : .bashrc {'présent' if bashrc.exists() else 'absent'}")

## Jokers et tailles de fichiers

Dans le dossier `serveur/etc` et tous ses sous-dossiers, quelle que soit leur profondeur :

1. comptez les fichiers (sans les dossiers) ;
2. affichez les chemins des fichiers `.conf`, relatifs à `serveur/etc` (par exemple `nginx/nginx.conf`), dans l'ordre alphabétique ;
3. calculez la taille totale de ces fichiers `.conf`, en octets.

**Documentation utile**

- [`Path.is_file`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.is_file)
- [`Path.relative_to`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.PurePath.relative_to)
- [`Path.stat`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.stat)
- [`os.stat_result`](https://docs.python.org/fr/3/library/os.html#os.stat_result)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`rglob("*")` parcourt toute l'arborescence, dossiers compris : `is_file()` ne garde que les fichiers. `rglob("*.conf")` ne donne que les `.conf`. `chemin.relative_to(etc)` retire le début `serveur/etc` d'un chemin, et `chemin.stat().st_size` donne sa taille en octets. Vous devez trouver 11 fichiers, dont 6 `.conf`, pour un total de 230 octets.

```python
from pathlib import Path

etc = Path("serveur/etc")
fichiers = [chemin for chemin in etc.rglob("*") if ...]
print(f"{len(fichiers)} fichiers")

confs = sorted(etc.rglob(...))
for conf in confs:
  print(...)
print(f"Taille des .conf : {sum(... for conf in confs)} octets")
```

### Solution

In [ ]:
from pathlib import Path

etc = Path("serveur/etc")
fichiers = [chemin for chemin in etc.rglob("*") if chemin.is_file()]
print(f"{len(fichiers)} fichiers")

confs = sorted(etc.rglob("*.conf"))
for conf in confs:
  print(conf.relative_to(etc))
print(f"Taille des .conf : {sum(conf.stat().st_size for conf in confs)} octets")

## Archiver des fichiers

Créez une archive `confs.tar.gz` qui contient tous les fichiers `.conf` de `serveur/etc`, rangés dans les mêmes sous-dossiers que dans `serveur/etc`. Procédez par étapes :

1. créez un dossier temporaire ;
2. copiez-y chaque fichier `.conf` à son chemin relatif à `serveur/etc`, en créant d'abord les dossiers qui le contiennent ;
3. créez l'archive à partir du dossier temporaire, avant la fin du bloc `with`, qui l'efface.

Affichez enfin les noms des fichiers que contient l'archive.

**Documentation utile**

- [`Path.mkdir`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.mkdir)
- [`shutil.make_archive`](https://docs.python.org/fr/3/library/shutil.html#shutil.make_archive)
- [`tarfile.open`](https://docs.python.org/fr/3/library/tarfile.html#tarfile.open)
- [`TarFile.getmembers`](https://docs.python.org/fr/3/library/tarfile.html#tarfile.TarFile.getmembers)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`mkdir(parents=True, exist_ok=True)` crée un dossier et ses parents, sans erreur s'il existe déjà. `shutil.copy2(source, destination)` copie un fichier avec ses métadonnées. `shutil.make_archive("confs", "gztar", root_dir=dossier)` crée `confs.tar.gz` avec le contenu de `dossier` et renvoie son nom. `tarfile.open(nom)` ouvre l'archive (dans un `with`), dont `getmembers()` donne les éléments, et `membre.isfile()` dit si c'en est un fichier. Vous devez trouver les 6 fichiers `.conf` de l'exercice précédent, préfixés par `./` : `./logrotate.conf`, `./nginx/nginx.conf`…

```python
import shutil
import tarfile
import tempfile
from pathlib import Path

etc = Path("serveur/etc")
with tempfile.TemporaryDirectory() as temporaire:
  for conf in etc.rglob("*.conf"):
    copie = Path(temporaire) / ...
    copie.parent.mkdir(...)
    shutil.copy2(conf, copie)
  archive = shutil.make_archive(...)

with tarfile.open(archive) as tar:
  for membre in tar.getmembers():
    if ...:
      print(membre.name)
```

### Solution

In [ ]:
import shutil
import tarfile
import tempfile
from pathlib import Path

etc = Path("serveur/etc")
with tempfile.TemporaryDirectory() as temporaire:
  for conf in etc.rglob("*.conf"):
    copie = Path(temporaire) / conf.relative_to(etc)
    copie.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(conf, copie)
  archive = shutil.make_archive("confs", "gztar", root_dir=temporaire)

with tarfile.open(archive) as tar:
  for membre in tar.getmembers():
    if membre.isfile():
      print(membre.name)

In [ ]:
# Variante avec shutil.copytree, qui copie tout l'arbre en ignorant les fichiers qui ne
# finissent pas par .conf : l'archive garde aussi les dossiers sans .conf (cron.d,
# sites-enabled), vides.
import shutil
import tempfile
from pathlib import Path


def ignore_sauf_confs(dossier: str, noms: list[str]) -> list[str]:
  return [
    nom for nom in noms if (Path(dossier) / nom).is_file() and not nom.endswith(".conf")
  ]


with tempfile.TemporaryDirectory() as temporaire:
  copie = Path(temporaire) / "etc"
  shutil.copytree("serveur/etc", copie, ignore=ignore_sauf_confs)
  print(shutil.make_archive("confs-copytree", "gztar", root_dir=copie))

## Dater des journaux

Le dossier `serveur/var/log` contient les journaux de `dpkg`, le gestionnaire de paquets : `dpkg.log`, le plus récent, et ses rotations successives, `dpkg.log.1`, puis `dpkg.log.2.gz` et `dpkg.log.3.gz`, déjà compressés.

Copiez-les dans un nouveau dossier `journaux-dates`, en les renommant `AAAA-MM-JJ_hh-mm-ss_dpkg.log.gz`, où la date est celle de leur dernière modification, en temps universel (UTC). Compressez au passage avec `gzip` les fichiers qui ne le sont pas encore (ceux dont le nom ne finit pas par `.gz`). Affichez enfin le contenu du dossier `journaux-dates`.

**Documentation utile**

- [`Path.stat`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.stat)
- [`datetime.fromtimestamp`](https://docs.python.org/fr/3/library/datetime.html#datetime.datetime.fromtimestamp)
- [`datetime.strftime`](https://docs.python.org/fr/3/library/datetime.html#datetime.datetime.strftime)
- [Codes de format de `strftime`](https://docs.python.org/fr/3/library/datetime.html#strftime-and-strptime-format-codes)
- [`gzip.open`](https://docs.python.org/fr/3/library/gzip.html#gzip.open)
- [`shutil.copyfileobj`](https://docs.python.org/fr/3/library/shutil.html#shutil.copyfileobj)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`serveur/var/log` contient d'autres journaux : `glob("dpkg.log*")` ne donne que ceux de `dpkg`. `chemin.stat().st_mtime` est la date de dernière modification, en secondes depuis le 1er janvier 1970, que `datetime.fromtimestamp(secondes, UTC)` convertit en date UTC. Son format est `"%Y-%m-%d_%H-%M-%S"` : attention, `%m` est le mois et `%M` les minutes. Pour compresser, ouvrez la source en lecture binaire (`"rb"`), la destination avec `gzip.open(destination, "wb")`, et `shutil.copyfileobj(source, destination)` copie l'une dans l'autre. Le premier fichier, le plus ancien, est `2026-09-21_06-25-57_dpkg.log.gz`, et le dernier `2026-10-12_06-25-03_dpkg.log.gz`.

```python
import gzip
import shutil
from datetime import UTC, datetime
from pathlib import Path

dates = Path("journaux-dates")
dates.mkdir(exist_ok=True)
for journal in Path("serveur/var/log").glob("dpkg.log*"):
  date = datetime.fromtimestamp(..., UTC)
  destination = dates / f"{date.strftime(...)}_dpkg.log.gz"
  if journal.suffix == ".gz":
    shutil.copy2(journal, destination)
  else:
    with journal.open("rb") as source, gzip.open(...) as compresse:
      ...

for chemin in sorted(dates.iterdir()):
  print(chemin.name)
```

### Solution

In [ ]:
import gzip
import shutil
from datetime import UTC, datetime
from pathlib import Path

dates = Path("journaux-dates")
dates.mkdir(exist_ok=True)
for journal in Path("serveur/var/log").glob("dpkg.log*"):
  date = datetime.fromtimestamp(journal.stat().st_mtime, UTC)
  destination = dates / f"{date.strftime('%Y-%m-%d_%H-%M-%S')}_dpkg.log.gz"
  if journal.suffix == ".gz":
    shutil.copy2(journal, destination)
  else:
    with journal.open("rb") as source, gzip.open(destination, "wb") as compresse:
      shutil.copyfileobj(source, compresse)

for chemin in sorted(dates.iterdir()):
  print(chemin.name)

## Protéger un secret

Écrivez dans `serveur/home/alice/secret.txt` une chaîne aléatoire de 20 lettres et chiffres, tirée avec le module `secrets`, fait pour cela (les tirages du module `random` sont prévisibles : il ne sert pas pour un secret). Le fichier doit être créé avec le seul droit d'écriture pour son propriétaire (`0o200`), puis, une fois le secret écrit, n'avoir plus que le droit de lecture pour son propriétaire (`0o400`). Affichez ses permissions après l'écriture, puis à la fin.

Un fichier en lecture seule ne peut plus être rouvert en écriture par son propriétaire : supprimez d'abord le fichier s'il existe, pour pouvoir relancer la cellule. Notez que Colab et le Codespace vous font travailler en `root`, que les permissions n'arrêtent pas.

**Documentation utile**

- [`secrets.choice`](https://docs.python.org/fr/3/library/secrets.html#secrets.choice)
- [`string.ascii_letters`](https://docs.python.org/fr/3/library/string.html#string.ascii_letters)
- [Paramètre `opener` de `open`](https://docs.python.org/fr/3/library/functions.html#open)
- [`os.open`](https://docs.python.org/fr/3/library/os.html#os.open)
- [`Path.chmod`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.chmod)
- [`Path.unlink`](https://docs.python.org/fr/3/library/pathlib.html#pathlib.Path.unlink)
- [`stat.filemode`](https://docs.python.org/fr/3/library/stat.html#stat.filemode)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`secrets.choice(alphabet)` tire un caractère de `alphabet`, ici `string.ascii_letters + string.digits`. `open` crée le fichier avec les permissions par défaut : pour choisir les siennes, on lui passe un `opener`, une fonction qui reçoit le chemin et les drapeaux d'ouverture et appelle `os.open(chemin, drapeaux, permissions)`. `unlink(missing_ok=True)` supprime un fichier, sans erreur s'il n'existe pas. `stat.filemode(chemin.stat().st_mode)` affiche les permissions comme `ls -l` : vous devez voir `--w-------`, puis `-r--------`.

```python
import os
import secrets
import stat
import string
from pathlib import Path

alphabet = string.ascii_letters + string.digits
valeur = "".join(... for _ in range(20))

secret = Path("serveur/home/alice/secret.txt")
secret.unlink(...)
with open(secret, "w", opener=lambda chemin, drapeaux: os.open(...)) as fichier:
  fichier.write(valeur)
print(stat.filemode(secret.stat().st_mode))
secret.chmod(...)
print(stat.filemode(secret.stat().st_mode))
```

### Solution

In [ ]:
import os
import secrets
import stat
import string
from pathlib import Path

alphabet = string.ascii_letters + string.digits
valeur = "".join(secrets.choice(alphabet) for _ in range(20))

secret = Path("serveur/home/alice/secret.txt")
secret.unlink(missing_ok=True)
with open(
  secret, "w", opener=lambda chemin, drapeaux: os.open(chemin, drapeaux, 0o200)
) as fichier:
  fichier.write(valeur)
print(stat.filemode(secret.stat().st_mode))
secret.chmod(0o400)
print(stat.filemode(secret.stat().st_mode))

## Variables d'environnement

1. La variable d'environnement `PATH` liste les dossiers où le shell cherche les commandes, séparés par `os.pathsep` (`:` sous Linux et macOS, `;` sous Windows). Affichez-les, un par ligne et numérotés à partir de 1, en signalant ceux qui n'existent pas sur la machine.
2. Lisez la variable `NIVEAU_LOG`, avec `"info"` pour valeur par défaut si elle n'existe pas, et affichez sa valeur.
3. Donnez-lui la valeur `"debug"` dans `os.environ`, puis vérifiez qu'une commande lancée ensuite la voit, avec la cellule `!echo $NIVEAU_LOG` : une commande hérite de l'environnement du programme qui la lance.

**Documentation utile**

- [`os.environ`](https://docs.python.org/fr/3/library/os.html#os.environ)
- [`os.pathsep`](https://docs.python.org/fr/3/library/os.html#os.pathsep)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`os.environ` se lit comme un dictionnaire : `os.environ["PATH"]` lève une `KeyError` si la variable n'existe pas, `os.environ.get(nom, défaut)` renvoie alors la valeur par défaut, et `os.environ[nom] = valeur` la modifie pour le programme et les commandes qu'il lance ensuite. `enumerate(..., start=1)` numérote à partir de 1. Dans une cellule, `!` lance une commande du shell, où `$NIVEAU_LOG` est la valeur de la variable : la cellule doit afficher `debug`.

```python
import os
from pathlib import Path

for numero, dossier in enumerate(os.environ["PATH"].split(...), start=1):
  absent = "" if Path(dossier).exists() else " (absent)"
  print(f"{numero}. {dossier}{absent}")

print(os.environ.get(...))
os.environ[...] = ...
```

### Solution

In [ ]:
import os
from pathlib import Path

for numero, dossier in enumerate(os.environ["PATH"].split(os.pathsep), start=1):
  absent = "" if Path(dossier).exists() else " (absent)"
  print(f"{numero}. {dossier}{absent}")

print(os.environ.get("NIVEAU_LOG", "info"))
os.environ["NIVEAU_LOG"] = "debug"

In [ ]:
!echo $NIVEAU_LOG

## Lancer des commandes

Écrivez une fonction `lance(commande)` qui exécute une commande, donnée comme une liste de chaînes, avec `subprocess.run`, et renvoie sa sortie standard sous forme de texte. Si la commande échoue (son code de retour n'est pas 0), ou si le programme n'existe pas, elle affiche le code de retour et le message d'erreur, et renvoie `None`. Utilisez-la ensuite pour :

1. compter les paquets installés sur la machine : les lignes de `dpkg -l` qui commencent par `ii` ;
2. afficher l'espace disponible sur le disque du répertoire courant : la quatrième colonne de la dernière ligne de `df -h .` ;
3. lister un dossier qui n'existe pas, avec `ls dossier-absent` ;
4. lancer un programme qui n'existe pas, `commande-absente`.

**Documentation utile**

- [`subprocess.run`](https://docs.python.org/fr/3/library/subprocess.html#subprocess.run)
- [`subprocess.CalledProcessError`](https://docs.python.org/fr/3/library/subprocess.html#subprocess.CalledProcessError)
- [`FileNotFoundError`](https://docs.python.org/fr/3/library/exceptions.html#FileNotFoundError)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`subprocess.run(commande, capture_output=True, text=True, check=True)` capture les sorties au lieu de les afficher (`text=True` les décode en chaînes) et lève une `subprocess.CalledProcessError` si le code de retour n'est pas 0 ; le résultat a les attributs `stdout` et `returncode`, l'exception `returncode` et `stderr`. Un programme introuvable lève `FileNotFoundError`. `splitlines()` découpe un texte en lignes, `split()` une ligne en colonnes. Colab compte plusieurs centaines de paquets ; `ls dossier-absent` finit avec le code 2.

```python
import subprocess


def lance(commande: list[str]) -> str | None:
  try:
    resultat = subprocess.run(commande, ...)
  except subprocess.CalledProcessError as erreur:
    print(f"{commande} : code de retour {...}, erreur : {...}")
    return None
  except FileNotFoundError:
    print(f"{commande} : programme introuvable")
    return None
  return resultat.stdout


paquets = lance(["dpkg", "-l"])
if paquets is not None:
  installes = [ligne for ligne in paquets.splitlines() if ...]
  print(f"{len(installes)} paquets installés")

disque = lance(["df", "-h", "."])
if disque is not None:
  print(f"Espace disponible : {...}")

lance(["ls", "dossier-absent"])
lance(["commande-absente"])
```

### Solution

In [ ]:
import subprocess


def lance(commande: list[str]) -> str | None:
  try:
    resultat = subprocess.run(commande, capture_output=True, text=True, check=True)
  except subprocess.CalledProcessError as erreur:
    print(
      f"{commande} : code de retour {erreur.returncode}, "
      f"erreur : {erreur.stderr.strip()}"
    )
    return None
  except FileNotFoundError:
    print(f"{commande} : programme introuvable")
    return None
  return resultat.stdout


paquets = lance(["dpkg", "-l"])
if paquets is not None:
  installes = [ligne for ligne in paquets.splitlines() if ligne.startswith("ii")]
  print(f"{len(installes)} paquets installés")

disque = lance(["df", "-h", "."])
if disque is not None:
  print(f"Espace disponible : {disque.splitlines()[-1].split()[3]}")

lance(["ls", "dossier-absent"])
lance(["commande-absente"])

## Un outil en ligne de commande

Écrivez le script `cherche.py`, un petit `grep` : `python cherche.py MOTIF [FICHIER...]` affiche les lignes des fichiers donnés qui contiennent la chaîne `MOTIF`, ou celles de l'entrée standard si aucun fichier n'est donné, pour qu'on puisse l'utiliser dans un tube (*pipe*) comme les commandes Unix. De plus :

- quand la variable d'environnement `CHERCHE_VERBEUX` vaut `1`, chaque ligne est précédée du nom de son fichier et de son numéro de ligne dans ce fichier, sous la forme `fichier:numéro:ligne` ;
- à la fin, le script écrit le nombre de lignes trouvées sur la sortie d'erreur, pour ne pas le mêler aux résultats qu'un tube passe à la commande suivante ;
- sans motif, il écrit son mode d'emploi sur la sortie d'erreur et s'arrête avec le code de retour 2 ; sinon, il s'arrête avec le code 0 s'il a trouvé une ligne, 1 sinon, comme `grep`.

La cellule d'exercice écrit le script avec `%%writefile` ; les trois cellules suivantes l'essaient sur un fichier, sur deux fichiers avec la variable `CHERCHE_VERBEUX`, puis dans un tube.

**Documentation utile**

- [`sys.argv`](https://docs.python.org/fr/3/library/sys.html#sys.argv)
- [`sys.exit`](https://docs.python.org/fr/3/library/sys.html#sys.exit)
- [`fileinput.input`](https://docs.python.org/fr/3/library/fileinput.html#fileinput.input)
- [`fileinput.filename`](https://docs.python.org/fr/3/library/fileinput.html#fileinput.filename)
- [`fileinput.filelineno`](https://docs.python.org/fr/3/library/fileinput.html#fileinput.filelineno)
- [`print`](https://docs.python.org/fr/3/library/functions.html#print)

In [ ]:
%%writefile cherche.py
# Votre code ici

In [ ]:
!python cherche.py error serveur/var/log/syslog

In [ ]:
!CHERCHE_VERBEUX=1 python cherche.py Failed serveur/var/log/auth.log serveur/var/log/syslog

In [ ]:
!cat serveur/var/log/syslog serveur/var/log/auth.log | python cherche.py sshd | wc -l

### Conseils supplémentaires

`sys.argv` est la liste des arguments de la ligne de commande, le nom du script compris : `sys.argv[1]` est le motif, `sys.argv[2:]` les fichiers. `fileinput.input(files=sys.argv[2:], encoding="utf8")` donne les lignes de ces fichiers à la suite, ou celles de l'entrée standard si la liste est vide ; pendant la boucle, `fileinput.filename()` et `fileinput.filelineno()` donnent le fichier et le numéro de la ligne courante. `print(..., file=sys.stderr)` écrit sur la sortie d'erreur, et `sys.exit(code)` arrête le script avec ce code de retour. Le premier essai trouve les 3 lignes `error` de `syslog`, le deuxième les 3 lignes `Failed` de `auth.log`, de `serveur/var/log/auth.log:1:` à `serveur/var/log/auth.log:3:`. Le troisième trouve 6 lignes : `wc -l`, qui compte les lignes qu'il reçoit par le tube, affiche 6, et le message « 6 ligne(s) trouvée(s) », écrit sur la sortie d'erreur, ne passe pas par le tube.

```python
%%writefile cherche.py
import fileinput
import os
import sys

if len(sys.argv) < 2:
  print(..., file=sys.stderr)
  sys.exit(2)

motif = sys.argv[1]
verbeux = os.environ.get(...) == "1"
trouvees = 0
for ligne in fileinput.input(files=..., encoding="utf8"):
  if ...:
    trouvees += 1
    if verbeux:
      print(f"{fileinput.filename()}:{...}:", end="")
    print(ligne, end="")
print(f"{trouvees} ligne(s) trouvée(s)", file=sys.stderr)
sys.exit(...)
```

### Solution

In [ ]:
%%writefile cherche.py
import fileinput
import os
import sys

if len(sys.argv) < 2:
  print("Usage : python cherche.py MOTIF [FICHIER...]", file=sys.stderr)
  sys.exit(2)

motif = sys.argv[1]
verbeux = os.environ.get("CHERCHE_VERBEUX") == "1"
trouvees = 0
for ligne in fileinput.input(files=sys.argv[2:], encoding="utf8"):
  if motif in ligne:
    trouvees += 1
    if verbeux:
      print(f"{fileinput.filename()}:{fileinput.filelineno()}:", end="")
    print(ligne, end="")
print(f"{trouvees} ligne(s) trouvée(s)", file=sys.stderr)
sys.exit(0 if trouvees else 1)